# 🍷 Wine Quality Prediction - End-to-End Machine Learning Project
**Course:** CSE570 - Machine Learning with Python

**Project Type:** Regression (with bonus Classification section)

**Dataset:** [Red Wine Quality - UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/wine+quality)

**Syllabus Coverage:** Unit 1 (Data Preparation) · Unit 2 (EDA) · Unit 3 (Regression) · Unit 4 (Classification - bonus)

## 📋 Problem Statement
- **Context:** A Portuguese wine company (Vinho Verde) wants to predict the quality of red wine based on physicochemical lab test results.
- **Target Variable:** `quality` — score between 3 and 8 (rated by wine experts)
- **ML Task:** Regression (predict continuous quality score)
- **Success Criteria:** Achieve R² > 0.3 and RMSE < 0.7 on test data
- **Intended Users:** Winemakers and quality control labs
- **Impact:** Reduce manual tasting costs, assist in production decisions and batch pricing

## 📊 Dataset Description
| Feature | Description |
|---|---|
| fixed acidity | Most acids involved with wine (g/dm³) |
| volatile acidity | Acetic acid amount - high levels → bad taste |
| citric acid | Adds freshness and flavor |
| residual sugar | Sugar remaining after fermentation |
| chlorides | Salt amount in wine |
| free sulfur dioxide | Prevents microbial growth |
| total sulfur dioxide | Total SO₂ (free + bound) |
| density | Density of wine (close to water) |
| pH | Acidity/basicity on scale 0-14 |
| sulphates | Wine additive (antioxidant) |
| alcohol | Alcohol percentage (%) |
| **quality** | **Target: score 3-8 (expert rating)** |

**Source:** P. Cortez et al. (2009), UCI ML Repository — 1,599 samples (1,358 after removing duplicates)

## Cell 1: Import Libraries (Unit 1)

In [1]:
# Unit 1: Data Preparation libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Unit 3: Regression models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import (mean_absolute_error, mean_squared_error, 
                             r2_score)

# Bonus (Unit 4 - optional section)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                             classification_report, roc_auc_score)

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print("✅ All libraries imported successfully")

✅ All libraries imported successfully


## Cell 2: Load Dataset (Unit 1)
The dataset loads directly from the UCI repository URL — **no manual download needed**.

In [2]:
# Unit 1: Data loading - directly from UCI repository URL (no download needed!)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')   # Note: this file uses ';' as separator

print(f"Dataset Shape: {df.shape}")
print(f"\nFeatures: {list(df.columns)}")
df.head()

Dataset Shape: (1599, 12)

Features: ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## Cell 3: Basic Data Inspection (Unit 1)

In [3]:
# Unit 1: Understanding the data
print("Data Types:\n")
print(df.dtypes)

print("\nDataset Info:")
df.info()

print("\nDescriptive Statistics (rounded to 2 decimals):")
df.describe().round(2)

Data Types:

fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density       

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.00,1599.00,1599.00,1599.00,1599.00,1599.00,1599.00,1599.00,1599.00,1599.00,1599.00,1599.00
mean,8.32,0.53,0.27,2.54,0.09,15.87,46.47,1.00,3.31,0.66,10.42,5.64
std,1.74,0.18,0.19,1.41,0.05,10.46,32.90,0.00,0.15,0.17,1.07,0.81
min,4.60,0.12,0.00,0.90,0.01,1.00,6.00,0.99,2.74,0.33,8.40,3.00
25%,7.10,0.39,0.09,1.90,0.07,7.00,22.00,1.00,3.21,0.55,9.50,5.00
50%,7.90,0.52,0.26,2.20,0.08,14.00,38.00,1.00,3.31,0.62,10.20,6.00
75%,9.20,0.64,0.42,2.60,0.09,21.00,62.00,1.00,3.40,0.73,11.10,6.00
max,15.90,1.58,1.00,15.50,0.61,72.00,289.00,1.00,4.01,2.00,14.90,8.00
